<a href="https://colab.research.google.com/github/pradhapmoorthi/CVND/blob/Trial/2_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Nanodegree

## Project: Image Captioning

---

In this notebook, you will train your CNN-RNN model.  

You are welcome and encouraged to try out many different architectures and hyperparameters when searching for a good model.

This does have the potential to make the project quite messy!  Before submitting your project, make sure that you clean up:
- the code you write in this notebook.  The notebook should describe how to train a single CNN-RNN architecture, corresponding to your final choice of hyperparameters.  You should structure the notebook so that the reviewer can replicate your results by running the code in this notebook.  
- the output of the code cell in **Step 2**.  The output should show the output obtained when training the model from scratch.

This notebook **will be graded**.  

Feel free to use the links below to navigate the notebook:
- [Step 1](#step1): Training Setup
- [Step 2](#step2): Train your Model
- [Step 3](#step3): (Optional) Validate your Model

<a id='step1'></a>
## Step 1: Training Setup

In this step of the notebook, you will customize the training of your CNN-RNN model by specifying hyperparameters and setting other options that are important to the training procedure.  The values you set now will be used when training your model in **Step 2** below.

You should only amend blocks of code that are preceded by a `TODO` statement.  **Any code blocks that are not preceded by a `TODO` statement should not be modified**.

### Task #1

Begin by setting the following variables:
- `batch_size` - the batch size of each training batch.  It is the number of image-caption pairs used to amend the model weights in each training step.
- `vocab_threshold` - the minimum word count threshold.  Note that a larger threshold will result in a smaller vocabulary, whereas a smaller threshold will include rarer words and result in a larger vocabulary.  
- `vocab_from_file` - a Boolean that decides whether to load the vocabulary from file.
- `embed_size` - the dimensionality of the image and word embeddings.  
- `hidden_size` - the number of features in the hidden state of the RNN decoder.  
- `num_epochs` - the number of epochs to train the model.  We recommend that you set `num_epochs=3`, but feel free to increase or decrease this number as you wish.  [This paper](https://arxiv.org/pdf/1502.03044.pdf) trained a captioning model on a single state-of-the-art GPU for 3 days, but you'll soon see that you can get reasonable results in a matter of a few hours!  (_But of course, if you want your model to compete with current research, you will have to train for much longer._)
- `save_every` - determines how often to save the model weights.  We recommend that you set `save_every=1`, to save the model weights after each epoch.  This way, after the `i`th epoch, the encoder and decoder weights will be saved in the `models/` folder as `encoder-i.pkl` and `decoder-i.pkl`, respectively.
- `print_every` - determines how often to print the batch loss to the Jupyter notebook while training.  Note that you **will not** observe a monotonic decrease in the loss function while training - this is perfectly fine and completely expected!  You are encouraged to keep this at its default value of `100` to avoid clogging the notebook, but feel free to change it.
- `log_file` - the name of the text file containing - for every step - how the loss and perplexity evolved during training.

If you're not sure where to begin to set some of the values above, you can peruse [this paper](https://arxiv.org/pdf/1502.03044.pdf) and [this paper](https://arxiv.org/pdf/1411.4555.pdf) for useful guidance!  **To avoid spending too long on this notebook**, you are encouraged to consult these suggested research papers to obtain a strong initial guess for which hyperparameters are likely to work best.  Then, train a single model, and proceed to the next notebook (**3_Inference.ipynb**).  If you are unhappy with your performance, you can return to this notebook to tweak the hyperparameters (and/or the architecture in **model.py**) and re-train your model.

### Question 1

**Question:** Describe your CNN-RNN architecture in detail.  With this architecture in mind, how did you select the values of the variables in Task 1?  If you consulted a research paper detailing a successful implementation of an image captioning model, please provide the reference.

**Answer:**


### (Optional) Task #2

Note that we have provided a recommended image transform `transform_train` for pre-processing the training images, but you are welcome (and encouraged!) to modify it as you wish.  When modifying this transform, keep in mind that:
- the images in the dataset have varying heights and widths, and
- if using a pre-trained model, you must perform the corresponding appropriate normalization.

### Question 2

**Question:** How did you select the transform in `transform_train`?  If you left the transform at its provided value, why do you think that it is a good choice for your CNN architecture?

**Answer:**

### Task #3

Next, you will specify a Python list containing the learnable parameters of the model.  For instance, if you decide to make all weights in the decoder trainable, but only want to train the weights in the embedding layer of the encoder, then you should set `params` to something like:
```
params = list(decoder.parameters()) + list(encoder.embed.parameters())
```

### Question 3

**Question:** How did you select the trainable parameters of your architecture?  Why do you think this is a good choice?

**Answer:**

### Task #4

Finally, you will select an [optimizer](http://pytorch.org/docs/master/optim.html#torch.optim.Optimizer).

### Question 4

**Question:** How did you select the optimizer used to train your model?

**Answer:**

## 1. Basic Setup

In [1]:
import os
import sys
import zipfile
import urllib.request
import subprocess
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# ==============================
# COCO ROOT DIRECTORY
# ==============================
COCO_ROOT = "/content/coco"

# ✅ CONFIRMED image locations
COCO_IMAGES_TRAIN = os.path.join(COCO_ROOT, "train2014")
COCO_IMAGES_VAL   = os.path.join(COCO_ROOT, "val2014")

# Annotations and COCO API
COCO_ANN_DIR = os.path.join(COCO_ROOT, "annotations")
COCOAPI_DIR  = os.path.join(COCO_ROOT, "cocoapi")

# Vocabulary (generated during training)
VOCAB_FILE   = os.path.join(COCO_ANN_DIR, "vocab.pkl")

## 2. Google Drive (Checkpoint Only)

In [3]:
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/CVND/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/caption_model.pth"

Mounted at /content/drive


## 3. Clone GitHub Repo (Trial branch)

In [4]:
REPO_URL = "https://github.com/pradhapmoorthi/CVND.git"
BRANCH = "Trial"
REPO_DIR = "/content/CVND"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(
        ["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR]
    )

sys.path.insert(0, REPO_DIR)

from model import EncoderCNN, DecoderRNN
from vocabulary import Vocabulary
from data_loader import get_loader

print("✅ Imported model.py, vocabulary.py, data_loader.py from GitHub (Trial branch)")

✅ Imported model.py, vocabulary.py, data_loader.py from GitHub (Trial branch)


In [5]:
!git clone https://github.com/cocodataset/cocoapi.git /content/coco/cocoapi


Cloning into '/content/coco/cocoapi'...
remote: Enumerating objects: 975, done.
remote: Total 975 (delta 0), reused 0 (delta 0), pack-reused 975 (from 1)
Receiving objects: 100% (975/975), 11.72 MiB | 16.74 MiB/s, done.
Resolving deltas: 100% (576/576), done.


In [6]:
!ls /content/coco/cocoapi


common	license.txt  LuaAPI  MatlabAPI	PythonAPI  README.txt  results


## 4. Dataset Download (Local)

In [7]:


# Download annotations
!wget -q -O /content/coco/annotations_trainval2014.zip \
    http://images.cocodataset.org/annotations/annotations_trainval2014.zip

# Extract
!unzip -q /content/coco/annotations_trainval2014.zip -d /content/coco

# Verify
!ls -lah /content/coco/annotations | head -n 20

total 806M
drwxr-xr-x 2 root root 4.0K Apr 20 18:25 .
drwxr-xr-x 4 root root 4.0K Apr 20 18:25 ..
-rw-rw-r-- 1 root root  64M Sep  1  2017 captions_train2014.json
-rw-rw-r-- 1 root root  31M Sep  1  2017 captions_val2014.json
-rw-rw-r-- 1 root root 318M Sep  1  2017 instances_train2014.json
-rw-rw-r-- 1 root root 154M Sep  1  2017 instances_val2014.json
-rw-r--r-- 1 root root 163M Sep  1  2017 person_keypoints_train2014.json
-rw-r--r-- 1 root root  78M Sep  1  2017 person_keypoints_val2014.json


In [ ]:

# Train images
!wget -q -O /content/coco/train2014.zip \
    http://images.cocodataset.org/zips/train2014.zip
!unzip -q /content/coco/train2014.zip -d /content/coco

# Val images
!wget -q -O /content/coco/val2014.zip \
    http://images.cocodataset.org/zips/val2014.zip
!unzip -q /content/coco/val2014.zip -d /content/coco


## 5. Training Configuration

In [ ]:
!ls /content/coco/annotations

In [ ]:
# ==============================
# REQUIRED FILE CHECKS (FAIL FAST)
# ==============================
assert os.path.exists(COCO_IMAGES_TRAIN), \
    f"❌ Missing folder: {COCO_IMAGES_TRAIN}"

assert os.path.exists(COCO_IMAGES_VAL), \
    f"❌ Missing folder: {COCO_IMAGES_VAL}"

assert os.path.exists(os.path.join(COCO_ANN_DIR, "captions_train2014.json")), \
    "❌ captions_train2014.json not found"

assert os.path.exists(os.path.join(COCO_ANN_DIR, "captions_val2014.json")), \
    "❌ captions_val2014.json not found"

# cocoapi must contain annotations directory
assert os.path.exists(os.path.join(COCOAPI_DIR, "annotations")), \
    "❌ cocoapi/annotations directory is missing"

print("✅ COCO directory structure validated")

In [ ]:

BATCH_SIZE = 64
LR = 3e-4
EPOCHS = 25

EMBED_SIZE = 512
ATTENTION_DIM = 512
HIDDEN_SIZE = 512
DROPOUT = 0.5

## 6. Data Loader (with Transform)

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406),
                         (0.229, 0.224, 0.225))])

In [ ]:
BATCH_SIZE = 64

train_loader = get_loader(
    transform=transform_train,
    mode="train",
    batch_size=BATCH_SIZE,
    vocab_threshold=5,
    vocab_file=VOCAB_FILE,
    start_word="<start>",
    end_word="<end>",
    unk_word="<unk>",
    vocab_from_file=False,
    num_workers=2,
    cocoapi_loc=COCOAPI_DIR
)

vocab = train_loader.dataset.vocab
print(f"✅ Vocabulary size: {len(vocab)}")

## 7. Model Initialization

In [ ]:
encoder = EncoderCNN(encoded_image_size=7).to(device)

decoder = DecoderRNN(
    attention_dim=ATTENTION_DIM,
    embed_size=EMBED_SIZE,
    hidden_size=HIDDEN_SIZE,
    vocab_size=len(vocab),
    dropout=DROPOUT,
).to(device)

## 8. Loss & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss(
    ignore_index=vocab.word2idx["<pad>"]
)

params = list(decoder.parameters()) + [
    p for p in encoder.parameters() if p.requires_grad
]

optimizer = optim.Adam(params, lr=LR)

## 9. Training Loop

In [ ]:
print("🚀 Starting training")

for epoch in range(EPOCHS):
    encoder.train()
    decoder.train()

    total_loss = 0.0

    for images, captions in train_loader:
        images = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        features = encoder(images)
        outputs, _ = decoder(features, captions)

        loss = criterion(
            outputs.reshape(-1, outputs.size(2)),
            captions[:, 1:].reshape(-1),
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f}")

## 10. Save Checkpoint to Google Drive

In [ ]:
torch.save(
    {
        "encoder": encoder.state_dict(),
        "decoder": decoder.state_dict(),
        "vocab": vocab,
    },
    CHECKPOINT_PATH,
)

print("✅ Training complete")
print(f"✅ Checkpoint saved to: {CHECKPOINT_PATH}")

<a id='step2'></a>
## Step 2: Train your Model

Once you have executed the code cell in **Step 1**, the training procedure below should run without issue.  

It is completely fine to leave the code cell below as-is without modifications to train your model.  However, if you would like to modify the code used to train the model below, you must ensure that your changes are easily parsed by your reviewer.  In other words, make sure to provide appropriate comments to describe how your code works!  

You may find it useful to load saved weights to resume training.  In that case, note the names of the files containing the encoder and decoder weights that you'd like to load (`encoder_file` and `decoder_file`).  Then you can load the weights by using the lines below:

```python
# Load pre-trained weights before resuming training.
encoder.load_state_dict(torch.load(os.path.join('./models', encoder_file)))
decoder.load_state_dict(torch.load(os.path.join('./models', decoder_file)))
```

While trying out parameters, make sure to take extensive notes and record the settings that you used in your various training runs.  In particular, you don't want to encounter a situation where you've trained a model for several hours but can't remember what settings you used :).

### A Note on Tuning Hyperparameters

To figure out how well your model is doing, you can look at how the training loss and perplexity evolve during training - and for the purposes of this project, you are encouraged to amend the hyperparameters based on this information.  

However, this will not tell you if your model is overfitting to the training data, and, unfortunately, overfitting is a problem that is commonly encountered when training image captioning models.  

For this project, you need not worry about overfitting. **This project does not have strict requirements regarding the performance of your model**, and you just need to demonstrate that your model has learned **_something_** when you generate captions on the test data.  For now, we strongly encourage you to train your model for the suggested 3 epochs without worrying about performance; then, you should immediately transition to the next notebook in the sequence (**3_Inference.ipynb**) to see how your model performs on the test data.  If your model needs to be changed, you can come back to this notebook, amend hyperparameters (if necessary), and re-train the model.

That said, if you would like to go above and beyond in this project, you can read about some approaches to minimizing overfitting in section 4.3.1 of [this paper](http://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=7505636).  In the next (optional) step of this notebook, we provide some guidance for assessing the performance on the validation dataset.

<a id='step3'></a>
## Step 3: (Optional) Validate your Model

To assess potential overfitting, one approach is to assess performance on a validation set.  If you decide to do this **optional** task, you are required to first complete all of the steps in the next notebook in the sequence (**3_Inference.ipynb**); as part of that notebook, you will write and test code (specifically, the `sample` method in the `DecoderRNN` class) that uses your RNN decoder to generate captions.  That code will prove incredibly useful here.

If you decide to validate your model, please do not edit the data loader in **data_loader.py**.  Instead, create a new file named **data_loader_val.py** containing the code for obtaining the data loader for the validation data.  You can access:
- the validation images at filepath `'/opt/cocoapi/images/train2014/'`, and
- the validation image caption annotation file at filepath `'/opt/cocoapi/annotations/captions_val2014.json'`.

The suggested approach to validating your model involves creating a json file such as [this one](https://github.com/cocodataset/cocoapi/blob/master/results/captions_val2014_fakecap_results.json) containing your model's predicted captions for the validation images.  Then, you can write your own script or use one that you [find online](https://github.com/tylin/coco-caption) to calculate the BLEU score of your model.  You can read more about the BLEU score, along with other evaluation metrics (such as TEOR and Cider) in section 4.1 of [this paper](https://arxiv.org/pdf/1411.4555.pdf).  For more information about how to use the annotation file, check out the [website](http://cocodataset.org/#download) for the COCO dataset.